# 03 - Causal patching (Phase 4), GPU  — FINAL version (v4)

**Settings:** Accelerator **GPU T4 x1** (x2 also fine) | Internet **On** | Input: the NEW bundle dataset | Secret `HF_TOKEN` (optional).
**Runtime:** ~5 min preflight, then ~2.3 h.  Start in a fresh session.

What it measures (51 token-aligned clean/counterfactual pairs; the two prompts differ ONLY in the swapped entity):
* **A** - copy the clean run's internal state into the counterfactual run at block L, either at the entity tokens,
  the tokens after the entity ("tail"), or only the last prompt token; does hop 1 now name the right bridge entity?
  Plus a donor-entity control (should NOT restore) and an all-positions sanity check (should restore ~100 %).
* **B** - transplant the single state at `</hop1>` (the old Phase-4 idea, now done deterministically).
* **C** - after each entity patch, does the hop-1 **probe** change its verdict? (links Phase 4 to the probe)

**In the preflight output (STEP 3), check these lines before letting the main run go:**
* `entity span length: median 5` (small numbers like 2-11; if you see hundreds, stop)
* `harmonized non-entity differences: ... e.g. '8' -> '7'` (the date token; fine)
* `HOOK SELF-TEST: ... -> PASS`
* `PROBE-LINK SELF-CHECK: cosine ... = [0.99..., ...] -> OK` (if it says WARNING, A and B are still valid; only C is doubtful)

Safety: the main run saves every pair to disk and resumes automatically (up to 3 tries); the last cell always packages
whatever exists (also visible in the Output tab under `results/patching/`).
Tip: after the preflight passes, use **Save Version -> Save & Run All (Commit)** so it runs in the background.

In [ ]:
# STEP 1 - locate the uploaded bundle and load the helper module.
import os, sys
from pathlib import Path
_inp = Path(os.environ.get("LF_KAGGLE_INPUT", "/kaggle/input"))
_hits = sorted(_inp.rglob("kaggle_helpers.py"))
assert _hits, ("Bundle not found. In the right panel click 'Add Input' and add the Kaggle Dataset you created "
               "from linguafranca_bundle.zip, then run this cell again.")
sys.path.insert(0, str(_hits[0].parent))
import kaggle_helpers as kh
WORK = kh.setup()

In [ ]:
# STEP 2 - login, GPU check, model reachability.
HF_TOKEN = kh.hf_login()
kh.require_gpu(min_total_gb=8)
MODEL = os.environ.get("LF_TEST_MODELS") or "meta-llama/Llama-3.2-3B-Instruct"
CHECK = kh.check_models([MODEL], HF_TOKEN)
assert CHECK[0][1], "Model not reachable - read the message above (licence not accepted? Internet off?)."

In [ ]:
# STEP 3 - PREFLIGHT: 2 pairs, 2 layers, includes the hook self-test. Stops here, loudly, if anything is wrong.
PRE = kh.RESULTS / "preflight_patching"
rc = kh.run([sys.executable, "causal_patching.py", "--data", "data/canonical/augmented_v2.jsonl", "--model", MODEL,
             "--max-pairs", "2", "--layers", "0,10", "--boundary-layers", "10", "--max-new-tokens", "40",
             "--answer-tokens", "60", "--out", str(PRE)], "preflight_patching")
assert rc == 0 and (PRE / "summary.md").exists(), "PREFLIGHT FAILED - see the log above (also in results/logs). Nothing long was started."
print("\nPREFLIGHT PASSED (hook self-test included) - starting the main run.")

In [ ]:
# STEP 4 - MAIN RUN (resumable: re-running this cell continues where it stopped).
OUT = kh.RESULTS / "patching"
LAYERS = os.environ.get("LF_TEST_LAYERS", "0,2,4,6,8,9,10,11,12,13,14,16,20,24")
extra = ["--max-pairs", os.environ["LF_TEST_MAX_PAIRS"]] if os.environ.get("LF_TEST_MAX_PAIRS") else []
rc = kh.run_with_retries([sys.executable, "causal_patching.py", "--data", "data/canonical/augmented_v2.jsonl",
                          "--model", MODEL, "--layers", LAYERS, "--out", str(OUT)] + extra, "patching", attempts=3)
print("main run exit code:", rc)

In [ ]:
# STEP 5 - FINAL (always runs): rebuild the summary from whatever pairs finished, show, package.
kh.run([sys.executable, "causal_patching.py", "--out", str(OUT), "--summarize-only"], "summarize")
kh.package(OUT, "patching_results")
print("Download patching_results.zip from the Output tab (or take files from results/patching/).")